In [1]:
import argparse
import json
import logging
from pathlib import Path
from typing import Optional, Union

import numpy as np
import polars as pl

# Default max mutual information bound
DEFAULT_MI = 1/2
NUM_TRIALS = 1000
NULL_VAL = 'null'


In [13]:
values = ['a', 'b', 'c', 'b', None]
mi =0.5
modified = []
for v in values:
    if v is None or isinstance(v, (float, np.floating)) and np.isnan(v):
        modified.append(NULL_VAL)
    else:
        modified.append(v)
        
print(modified)


['a', 'b', 'c', 'b', 'null']


In [14]:
categories, encoded = np.unique(modified, return_inverse=True)
cat_to_idx = {cat: i for i, cat in enumerate(categories)}
idx_to_cat = {i: cat for i, cat in enumerate(categories)}
one_hot_encodings = np.eye(len(categories))[encoded]

In [15]:
one_hot_encodings

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 1.]])

In [16]:
dims = one_hot_encodings.shape[1]
variances_per_dim = np.var(one_hot_encodings, axis=0)
assert dims == len(variances_per_dim)
sqrt_total_var = sum([variances_per_dim[x]**0.5 for x in range(len(variances_per_dim))])
per_dim_scale = [1./(2*mi) * variances_per_dim[ind]**0.5 * sqrt_total_var for ind in range(dims)]

In [17]:
variances_per_dim

array([0.16, 0.24, 0.16, 0.16])

In [18]:
per_dim_scale

[0.6759591794226544,
 0.8278775382679628,
 0.6759591794226544,
 0.6759591794226543]

In [21]:
releases = []
for _ in range(1):
    sample = np.random.choice(modified)
    print(sample)
    one_hot_rep = np.zeros(len(cat_to_idx))
    one_hot_rep[cat_to_idx[sample]] = 1
    print(one_hot_rep)
    for dim_ind in range(len(one_hot_rep)):
        one_hot_rep[dim_ind] += np.random.normal(loc=0, scale = np.sqrt(per_dim_scale[dim_ind]))
    print(one_hot_rep)
    release = idx_to_cat[np.argmax(one_hot_rep)]
    print(release)
    if release == NULL_VAL:
        releases.append(None)
    else:
        releases.append(release)

c
[0. 0. 1. 0.]
[-0.67187671  0.32735044  1.1428798  -0.03493593]
c


In [22]:
cat_to_idx

{'a': 0, 'b': 1, 'c': 2, 'null': 3}

In [23]:
idx_to_cat

{0: 'a', 1: 'b', 2: 'c', 3: 'null'}